# 📊 Sesión 1: Herramientas de Visualización de Datos
## Notebook Scala/Spark — Parte A (obligatoria)

**Kernel:** Almond (Scala 2.13.18) 
**Spark:** 4.1.1 
**Ruta de salida CSV:** `C:/Curso-Scala/datos/`

> ⚠️ Este notebook solo contiene celdas Scala. Para los gráficos (parte opcional) abre `clase_24_graficos.ipynb` con kernel Python.

---
## ⚙️ Celda 1 — Dependencias Ivy
Ejecuta esta celda primero. Tarda ~1 minuto la primera vez.

In [1]:
import $ivy.`org.apache.spark::spark-core:4.1.1`
import $ivy.`org.apache.spark::spark-sql:4.1.1`

import $ivy.$
import $ivy.$

---
## ⚙️ Celda 2 — SparkSession e imports

In [2]:
import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("Dia21_Sesion2_Visualizacion")
  .master("local[*]")
  .config("spark.sql.shuffle.partitions", "4")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")
println(s"✅ Spark ${spark.version} | Scala ${scala.util.Properties.versionString}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/07 22:00:10 INFO SparkContext: Running Spark version 4.1.1
26/05/07 22:00:10 INFO SparkContext: OS info Windows 11, 10.0, amd64
26/05/07 22:00:10 INFO SparkContext: Java version 17.0.18+8-LTS
26/05/07 22:00:10 INFO ResourceUtils: ==============================================================
26/05/07 22:00:10 INFO ResourceUtils: No custom resources configured for spark.driver.
26/05/07 22:00:10 INFO ResourceUtils: ==============================================================
26/05/07 22:00:10 INFO SparkContext: Submitted application: Dia21_Sesion2_Visualizacion
26/05/07 22:00:10 INFO SecurityManager: Changing view acls to: rodba
26/05/07 22:00:10 INFO SecurityManager: Changing modify acls to: rodba
26/05/07 22:00:10 INFO SecurityManager: Changing view acls groups to: rodba
26/05/07 22:00:10 INFO SecurityManager: Changing modify acls groups to: rodba
26/05/07 22:00:10 INFO SecurityManager: SecurityMa

✅ Spark 4.1.1 | Scala version 2.13.18


import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._
spark: SparkSession = org.apache.spark.sql.classic.SparkSession@680ccf16
import spark.implicits._

---
## 📦 Celda 3 — Dataset de trabajo
Ventas de una tienda de tecnología y oficina durante 15 días de abril 2026.

In [3]:
val ventasRaw = Seq(
  (1,  "2026-04-01", "Tecnología", "Laptop",      1200.0,  2, "Madrid"),
  (2,  "2026-04-01", "Oficina",    "Silla",         120.0,  4, "Barcelona"),
  (3,  "2026-04-02", "Tecnología", "Monitor",       350.0,  3, "Valencia"),
  (4,  "2026-04-02", "Oficina",    "Mesa",           250.0,  1, "Madrid"),
  (5,  "2026-04-03", "Tecnología", "Teclado",         45.0,  8, "Sevilla"),
  (6,  "2026-04-03", "Tecnología", "Ratón",           25.0, 12, "Madrid"),
  (7,  "2026-04-04", "Oficina",    "Archivador",      80.0,  3, "Valencia"),
  (8,  "2026-04-04", "Tecnología", "Laptop",        1200.0,  1, "Barcelona"),
  (9,  "2026-04-05", "Tecnología", "Tablet",         600.0,  2, "Madrid"),
  (10, "2026-04-05", "Oficina",    "Silla",          120.0,  6, "Sevilla"),
  (11, "2026-04-06", "Tecnología", "Monitor",        350.0,  2, "Madrid"),
  (12, "2026-04-06", "Tecnología", "Impresora",      200.0,  1, "Barcelona"),
  (13, "2026-04-07", "Oficina",    "Mesa",           250.0,  2, "Valencia"),
  (14, "2026-04-07", "Tecnología", "Teclado",         45.0,  5, "Madrid"),
  (15, "2026-04-08", "Tecnología", "Laptop",        1200.0,  3, "Sevilla"),
  (16, "2026-04-08", "Oficina",    "Silla",          120.0,  2, "Madrid"),
  (17, "2026-04-09", "Tecnología", "Ratón",           25.0, 10, "Barcelona"),
  (18, "2026-04-09", "Tecnología", "Tablet",         600.0,  1, "Valencia"),
  (19, "2026-04-10", "Oficina",    "Archivador",      80.0,  4, "Madrid"),
  (20, "2026-04-10", "Tecnología", "Monitor",        350.0,  2, "Sevilla"),
  (21, "2026-04-11", "Tecnología", "Laptop",        1200.0,  2, "Madrid"),
  (22, "2026-04-11", "Oficina",    "Mesa",           250.0,  3, "Barcelona"),
  (23, "2026-04-12", "Tecnología", "Teclado",         45.0,  6, "Valencia"),
  (24, "2026-04-12", "Tecnología", "Impresora",      200.0,  2, "Madrid"),
  (25, "2026-04-13", "Oficina",    "Silla",          120.0,  5, "Sevilla"),
  (26, "2026-04-13", "Tecnología", "Ratón",           25.0, 15, "Madrid"),
  (27, "2026-04-14", "Tecnología", "Monitor",        350.0,  1, "Barcelona"),
  (28, "2026-04-14", "Oficina",    "Archivador",      80.0,  2, "Valencia"),
  (29, "2026-04-15", "Tecnología", "Tablet",         600.0,  2, "Madrid"),
  (30, "2026-04-15", "Tecnología", "Laptop",        1200.0,  1, "Barcelona")
)

val dfVentas = ventasRaw.toDF(
  "id", "fecha", "categoria", "producto",
  "precio_unitario", "cantidad", "ciudad"
).withColumn("total_venta", round(col("precio_unitario") * col("cantidad"), 2))

dfVentas.show(5)
println(s"Total registros: ${dfVentas.count()}")

+---+----------+----------+--------+---------------+--------+---------+-----------+
| id|     fecha| categoria|producto|precio_unitario|cantidad|   ciudad|total_venta|
+---+----------+----------+--------+---------------+--------+---------+-----------+
|  1|2026-04-01|Tecnología|  Laptop|         1200.0|       2|   Madrid|     2400.0|
|  2|2026-04-01|   Oficina|   Silla|          120.0|       4|Barcelona|      480.0|
|  3|2026-04-02|Tecnología| Monitor|          350.0|       3| Valencia|     1050.0|
|  4|2026-04-02|   Oficina|    Mesa|          250.0|       1|   Madrid|      250.0|
|  5|2026-04-03|Tecnología| Teclado|           45.0|       8|  Sevilla|      360.0|
+---+----------+----------+--------+---------------+--------+---------+-----------+
only showing top 5 rows
Total registros: 30


ventasRaw: Seq[(Int, String, String, String, Double, Int, String)] = List(
  (1, "2026-04-01", "Tecnología", "Laptop", 1200.0, 2, "Madrid"),
  (2, "2026-04-01", "Oficina", "Silla", 120.0, 4, "Barcelona"),
  (3, "2026-04-02", "Tecnología", "Monitor", 350.0, 3, "Valencia"),
  (4, "2026-04-02", "Oficina", "Mesa", 250.0, 1, "Madrid"),
  (5, "2026-04-03", "Tecnología", "Teclado", 45.0, 8, "Sevilla"),
  (6, "2026-04-03", "Tecnología", "Ratón", 25.0, 12, "Madrid"),
  (7, "2026-04-04", "Oficina", "Archivador", 80.0, 3, "Valencia"),
  (8, "2026-04-04", "Tecnología", "Laptop", 1200.0, 1, "Barcelona"),
  (9, "2026-04-05", "Tecnología", "Tablet", 600.0, 2, "Madrid"),
  (10, "2026-04-05", "Oficina", "Silla", 120.0, 6, "Sevilla"),
  (11, "2026-04-06", "Tecnología", "Monitor", 350.0, 2, "Madrid"),
  (12, "2026-04-06", "Tecnología", "Impresora", 200.0, 1, "Barcelona"),
  (13, "2026-04-07", "Oficina", "Mesa", 250.0, 2, "Valencia"),
  (14, "2026-04-07", "Tecnología", "Teclado", 45.0, 5, "Madrid"),
  (15

---
## ✅ P1 — Exportar ventas por categoría a CSV

Agrupa por categoría y guarda el resumen. 
El CSV resultante se usará en el ejercicio opcional P4 (gráfico de barras).

In [4]:
val dfPorCategoria = dfVentas
  .groupBy("categoria")
  .agg(
    round(sum("total_venta"), 2).alias("total_ventas"),
    count("*").alias("num_transacciones"),
    round(avg("precio_unitario"), 2).alias("precio_medio")
  )
  .orderBy(col("total_ventas").desc)

dfPorCategoria.show()

dfPorCategoria
  .coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv("C:/Curso-Scala/datos/viz_por_categoria")

println("✅ CSV exportado → C:/Curso-Scala/datos/viz_por_categoria")

+----------+------------+-----------------+------------+
| categoria|total_ventas|num_transacciones|precio_medio|
+----------+------------+-----------------+------------+
|Tecnología|     18980.0|               20|       490.5|
|   Oficina|      4260.0|               10|       147.0|
+----------+------------+-----------------+------------+

✅ CSV exportado → C:/Curso-Scala/datos/viz_por_categoria


dfPorCategoria: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [categoria: string, total_ventas: double ... 2 more fields]

---
## ✅ P2 — Exportar evolución temporal de ventas diarias

Agrupa por fecha y guarda el total por día. 
El CSV resultante se usará en el ejercicio opcional P5 (gráfico de líneas).

In [5]:
val dfPorFecha = dfVentas
  .groupBy("fecha")
  .agg(round(sum("total_venta"), 2).alias("total_dia"))
  .orderBy("fecha")

dfPorFecha.show(10)

dfPorFecha
  .coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv("C:/Curso-Scala/datos/viz_por_fecha")

println("✅ CSV exportado → C:/Curso-Scala/datos/viz_por_fecha")

+----------+---------+
|     fecha|total_dia|
+----------+---------+
|2026-04-01|   2880.0|
|2026-04-02|   1300.0|
|2026-04-03|    660.0|
|2026-04-04|   1440.0|
|2026-04-05|   1920.0|
|2026-04-06|    900.0|
|2026-04-07|    725.0|
|2026-04-08|   3840.0|
|2026-04-09|    850.0|
|2026-04-10|   1020.0|
+----------+---------+
only showing top 10 rows
✅ CSV exportado → C:/Curso-Scala/datos/viz_por_fecha


dfPorFecha: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [fecha: string, total_dia: double]

---
## ✅ P3 — Exportar Top 5 productos por ventas

Obtiene el ranking de los 5 productos más vendidos. 
El CSV resultante se usará en el ejercicio opcional P6 (dashboard).

In [6]:
val dfTopProductos = dfVentas
  .groupBy("producto")
  .agg(round(sum("total_venta"), 2).alias("total"))
  .orderBy(col("total").desc)
  .limit(5)

dfTopProductos.show()

dfTopProductos
  .coalesce(1)
  .write
  .option("header", "true")
  .mode("overwrite")
  .csv("C:/Curso-Scala/datos/viz_top_productos")

println("✅ CSV exportado → C:/Curso-Scala/datos/viz_top_productos")

+--------+-------+
|producto|  total|
+--------+-------+
|  Laptop|10800.0|
|  Tablet| 3000.0|
| Monitor| 2800.0|
|   Silla| 2040.0|
|    Mesa| 1500.0|
+--------+-------+

✅ CSV exportado → C:/Curso-Scala/datos/viz_top_productos


dfTopProductos: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row] = [producto: string, total: double]

---
## 🏁 Fin de la Parte A

Si has llegado aquí con los tres CSV generados, has completado la parte obligatoria.

Verifica que existen las carpetas en `C:\Curso-Scala\datos\`:
- `viz_por_categoria\`
- `viz_por_fecha\`
- `viz_top_productos\`

Si tienes conocimientos de Python, abre ahora **`clase_24_graficos.ipynb`** para visualizar los resultados.

In [7]:
// Verificación final: listar los CSV generados
import java.io.File

val carpetas = List(
  "C:/Curso-Scala/datos/viz_por_categoria",
  "C:/Curso-Scala/datos/viz_por_fecha",
  "C:/Curso-Scala/datos/viz_top_productos"
)

carpetas.foreach { ruta =>
  val dir = new File(ruta)
  if (dir.exists()) {
    val csv = dir.listFiles().filter(_.getName.endsWith(".csv"))
    println(s"✅ $ruta  →  ${csv.map(_.getName).mkString(", ")}")
  } else {
    println(s"❌ NO encontrada: $ruta")
  }
}

✅ C:/Curso-Scala/datos/viz_por_categoria  →  part-00000-56da49da-4adb-43a0-a340-776653bc4971-c000.csv
✅ C:/Curso-Scala/datos/viz_por_fecha  →  part-00000-f334d6e9-c309-4680-b34e-ec89107df3fc-c000.csv
✅ C:/Curso-Scala/datos/viz_top_productos  →  part-00000-ac52d93d-b860-42bc-9bcb-ecca440cd243-c000.csv


import java.io.File
carpetas: List[String] = List(
  "C:/Curso-Scala/datos/viz_por_categoria",
  "C:/Curso-Scala/datos/viz_por_fecha",
  "C:/Curso-Scala/datos/viz_top_productos"
)